# 自定义中间件 - Wrap-style hooks
## wrap_model_call 装饰器实现

In [14]:

from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware, \
    ExtendedModelResponse
from langchain.agents.middleware.types import ResponseT
from langchain_core.messages import SystemMessage, AIMessage
from langgraph.typing import ContextT


@wrap_model_call
def wrap_model_call(request: ModelRequest,
                    handler: Callable[[ModelRequest], ModelResponse], ) -> ModelResponse | None:
    system_prompt = request.system_prompt if request.system_prompt else ''

    new_system_prompt = f"""
        {system_prompt}
        用户所在地域：American
        用户所选语言：English
        如果用户未特意强调，即便他用中文回复你，你也要根据用户所在地区和所选用语言进行回复
    """
    request = request.override(
        system_message=SystemMessage(content=new_system_prompt),
    )

    return handler(request)

## wrap_model_call类实现

In [12]:
class RulesMiddleware(AgentMiddleware):

    def wrap_model_call(
            self,
            request: ModelRequest[ContextT],
            handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
    ) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:
        system_prompt = request.system_prompt if request.system_prompt else ''

        new_system_prompt = f"""
            {system_prompt}
            用户所在地域：American
            用户所选语言：English
            如果用户未特意强调，即便他用中文回复你，你也要根据用户所在地区和所选用语言进行回复
        """

        request = request.override(
            system_message=SystemMessage(content=new_system_prompt)
        )

        print(">>> 实际发给模型的 system_prompt:")
        print(request.system_prompt)
        return handler(request)

In [15]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from rich import print as rprint

from common import init_simple_qwen_max

model = init_simple_qwen_max()

agent = create_agent(
    model=model,
    middleware=[
        # RulesMiddleware(),
        wrap_model_call
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("简单介绍一下中国历史")]
})

# rprint(response)
#
for msg in response["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='简单介绍一下中国历史',
            additional_kwargs={},
            response_metadata={},
            id='86364595-6685-4bc8-b2d8-3b4b9089625a'
        ),
        AIMessage(
            content="Chinese history is a vast and intricate subject, spanning thousands of years. Here's a brief 
overview to give you an idea of its key periods and developments:\n\n1. **Prehistoric Period (Before 2070 BC)**: 
This era includes the Neolithic cultures such as Yangshao and Longshan, which laid the foundation for early Chinese
civilization.\n\n2. **Xia Dynasty (c. 2070–1600 BC)**: Often considered the first dynasty in Chinese history, 
although it is more legendary than historically verified. It was succeeded by the Shang Dynasty.\n\n3. **Shang 
Dynasty (c. 1600–1046 BC)**: Known for its advanced bronze metallurgy and the earliest known form of Chinese 
writing, inscribed on oracle bones.\n\n4. **Zhou Dynasty (1046–256 BC)**: The longest-lasting dynasty in Chinese 
history, divided into Western Zhou and Eastern Zhou. The latter period saw the rise of many small states and the 
development of major philosophical schools, including Confucianism, Taoism, and Legalism.\n\n5. **Qin Dynasty 
(221–206 BC)**: Unified China under a centralized state, standardized weights, measures, and currency, and began 
construction of the Great Wall.\n\n6. **Han Dynasty (206 BC–220 AD)**: A golden age of Chinese culture, with 
significant advancements in science, technology, and art. The Silk Road was established during this period, 
facilitating trade and cultural exchange.\n\n7. **Three Kingdoms (220–280 AD)**: A time of division and conflict, 
followed by the Jin Dynasty, which also faced internal strife and invasions from the north.\n\n8. **Northern and 
Southern Dynasties (420–589 AD)**: A period of political fragmentation, but also one of cultural and artistic 
development, particularly in the south.\n\n9. **Sui Dynasty (581–618 AD)**: Reunified China after a long period of 
division and initiated several large-scale projects, including the Grand Canal.\n\n10. **Tang Dynasty (618–907 
AD)**: Another golden age, marked by prosperity, cultural flourishing, and international influence. It is often 
considered the height of classical Chinese civilization.\n\n11. **Five Dynasties and Ten Kingdoms (907–960 AD)**: A
brief period of disunity, followed by the reunification under the Song Dynasty.\n\n12. **Song Dynasty (960–1279 
AD)**: Known for its technological innovations, economic growth, and cultural achievements, including the 
development of the civil service examination system.\n\n13. **Yuan Dynasty (1271–1368 AD)**: Founded by Kublai 
Khan, grandson of Genghis Khan, this was the first foreign-led dynasty to rule all of China, bringing Mongol 
influence to the region.\n\n14. **Ming Dynasty (1368–1644 AD)**: Known for its maritime expeditions, the 
construction of the Forbidden City, and the rebuilding of the Great Wall.\n\n15. **Qing Dynasty (1644–1912 AD)**: 
The last imperial dynasty, founded by the Manchus, which saw both great territorial expansion and internal 
challenges, leading to its eventual collapse and the establishment of the Republic of China.\n\n16. **Republic of 
China (1912–1949) and People's Republic of China (1949–Present)**: The modern era, marked by significant political,
social, and economic changes, including the rise of the Communist Party and the rapid modernization and economic 
growth of recent decades.\n\nThis brief overview covers the major dynasties and periods, but each has a rich and 
complex history that can be explored in much greater detail.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 815,
                    'prompt_tokens': 61,
                    'total_tokens': 876,
                    'completion_tokens_details': None,
                    'prompt_tokens_

## wrap_tool_call - 装饰器

In [16]:
from langchain_core.tools import tool


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res

In [27]:
import time
from datetime import datetime
from langgraph.types import Command
from typing import Any
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def tool_timer_middleware(request: ToolCallRequest,
                          handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                          ) -> ToolMessage | Command[Any]:
    tool_name = request.tool_call['name']
    start = time.perf_counter()
    print(f'[{tool_name}] start: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")[:-3]}')
    response = handler(request)
    end = time.perf_counter()
    elapsed = end - start
    print(f'[{tool_name}] end: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")[-3]}  (elapsed: {elapsed:.3f}s)')
    return response

## wrap_tool_call - 类实现

In [26]:
import time
from datetime import datetime

class ToolTimerMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]]) -> ToolMessage | Command[Any]:
        tool_name = request.tool_call['name']
        start = time.perf_counter()
        print(f'[{tool_name}] start: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")[:-3]}')
        response = handler(request)
        end = time.perf_counter()
        elapsed = end - start
        print(f'[{tool_name}] end: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")[-3]}  (elapsed: {elapsed:.3f}s)')
        return response


In [30]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from rich import print as rprint

from common import init_simple_qwen_max

model = init_simple_qwen_max()

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # tool_timer_middleware
        ToolTimerMiddleware()
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("查询一下今天北京的天气")]
})

# rprint(response)
#
for msg in response["messages"]:
    msg.pretty_print()

[get_weather] start: 2026-07-23 19:17
[get_weather] end: :  (elapsed: 0.004s)
================================ Human Message =================================

查询一下今天北京的天气
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_f088f30b904a403088ae98)
 Call ID: call_f088f30b904a403088ae98
  Args:
    city: 北京
    is_forcast: False
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
================================== Ai Message ==================================

今天北京的天气不错。
